# Passo 0: inspeção da base HYAMD

A lógica fica em `macdeg/exploration/inspection.py`. Este notebook só executa e mostra.

**Dados restritos (PhysioNet):** as saídas deste notebook podem conter imagens e IDs. Não compartilhe o arquivo com saídas; o filtro do Git as remove nos commits.

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'hyamd').exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from macdeg.exploration.centering import contact_sheet_by_width, run_centering
from macdeg.exploration.inspection import run_inspection

## Relatório agregado

A primeira execução lê as 1560 imagens e leva alguns minutos; depois reaproveita os arquivos de `outputs/step0/`.

In [ ]:
report = run_inspection(workers=6)
centering = run_centering(workers=6)
sheet_path = contact_sheet_by_width()
print(json.dumps(report, indent=1, default=str))
print(json.dumps(centering, indent=1, default=str))

## Inspeção visual: 15 imagens de cada formato

Cada imagem aparece com: cruz branca (centro do campo), círculo ciano (disco), amarelo (fóvea pela mancha escura) e magenta (fóvea prevista por disco + vetor médio). As 3 primeiras linhas têm 1960 px de largura; as 3 últimas, 2576 px.

**Verifique:**
1. O ciano está sobre o disco?
2. O amarelo e o magenta estão sobre a fóvea? Qual erra menos?
3. Nas imagens de 1960 px a fóvea fica deslocada do centro, e nas de 2576 px fica perto da cruz?

In [ ]:
sheet = np.asarray(Image.open(sheet_path))[..., ::-1]
plt.figure(figsize=(16, 19))
plt.imshow(sheet)
plt.axis('off')
plt.show()

## Posição do disco e da fóvea em toda a base

Coordenadas em unidades do raio do campo, com a origem no centro e o disco espelhado para o lado direito.

In [ ]:
df = pd.read_csv('outputs/step0/per_image_centering.csv')
fig, axes = plt.subplots(2, 2, figsize=(10, 10), sharex=True, sharey=True)
for row, (width, group) in zip(axes, df.groupby('width')):
    for ax, kind, title in zip(row, ['disc2', 'fovea2'], ['Disco', 'Fóvea (mancha escura)']):
        ax.scatter(group[f'{kind}_cdx'], group[f'{kind}_dy'], s=4, alpha=0.4, color='#1f77b4')
        ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='0.5'))
        ax.axhline(0, color='0.85', lw=0.8)
        ax.axvline(0, color='0.85', lw=0.8)
        ax.set(title=f'{title}, largura {width} px (n={len(group)})', xlim=(-1.05, 1.05), ylim=(1.05, -1.05), aspect='equal')
for ax in axes[1]:
    ax.set_xlabel('dx espelhado (raios do campo)')
for ax in axes[:, 0]:
    ax.set_ylabel('dy (raios do campo)')
plt.tight_layout()
plt.show()